In [ ]:
import numpyro

numpyro.set_host_device_count(4)

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive

print("Devices:", jax.local_device_count())

## Define and show model

In [ ]:
def one_throw_model(data: jax.Array | None = None) -> None:
    p = numpyro.sample("p", dist.Beta(1.0, 1.0))

    numpyro.sample("obs", dist.Bernoulli(p), obs=data)

In [ ]:
numpyro.render_model(one_throw_model, render_distributions=True)

## Sample from my model without data

In [ ]:
key = jax.random.PRNGKey(0)

predictive = Predictive(one_throw_model, num_samples=10)
samples = predictive(key)

print(samples)

## Several iid coin throws

In [ ]:
def n_throws_model(data: jax.Array | None = None, n: int = 10) -> None:
    p = numpyro.sample("p", dist.Beta(1.0, 1.0))

    with numpyro.plate("throws", n):
        numpyro.sample("obs", dist.Bernoulli(p), obs=data)

In [ ]:
numpyro.render_model(n_throws_model, render_distributions=True)

In [ ]:
key = jax.random.PRNGKey(0)

predictive = Predictive(n_throws_model, num_samples=5)
samples = predictive(key, n=10)
print(samples)

## Doing inference from data

In [ ]:
data = jnp.array([0, 1, 0, 1, 1, 0, 1, 0, 1, 1])

In [ ]:
kernel = NUTS(n_throws_model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=10000, num_chains=4,)
mcmc.run(jax.random.PRNGKey(0), data=data, n=len(data))
mcmc.print_summary()

In [ ]:
posterior_samples = mcmc.get_samples()

In [ ]:
posterior_samples

In [ ]:

plt.figure()
plt.hist(posterior_samples["p"], bins=100, density=True, alpha=0.7)
plt.xlabel("p")
plt.ylabel("density")
plt.show()

## Sample from the posterior predictive distribution

In [ ]:
posterior_predictive = Predictive(n_throws_model, posterior_samples=posterior_samples)
new_samples = posterior_predictive(jax.random.PRNGKey(1), n=10)